1.Embedding hóa tài liệu : Biến mỗi câu trong file csv thành các vcetor. Xuất embedding này sang .parquet và .npy 

In [ ]:
import pandas 
import numpy 
from sentence_transformers import SentenceTransformer

def read_sentence(file_path="data.csv"):
    dataFrame = pandas.read_csv(file_path)
    return dataFrame

#Embedding model hỗ trợ tốt tiếng Việt "keepitreal/vietnamese-sbert, device=0 -> chạy bằng GPU. 
def create_embedding(listSentence,modelName="keepitreal/vietnamese-sbert"):
    model = SentenceTransformer(modelName,device=0)
    embedding = model.encode(listSentence,show_progress_bar=True)
    return embedding,model

def save_embedding(dataFrame,embedding,path="embedding_sentence.parquet"):
    dataFrame = dataFrame.copy()
    dataFrame['embedding'] = list(embedding)
    dataFrame.to_parquet(path,index=False)
    numpy.save("genai-section5-lab",embedding)
    return dataFrame

if __name__ == "__main__":
    dataFrame = read_sentence()
    listSentence = dataFrame['cau'].tolist()
    embedding, model = create_embedding(listSentence)
    dataFrameResult = save_embedding(dataFrame,embedding)

    print("Mỗi một câu là một ma trận số.")
    print(f"Kích thước của ma trận embedding là: {embedding.shape}")

2.So sánh độ tương đồng giữa Dot Product và Cosine similarity : Lấy ra ma trận embedding, tính tích vô hướng và cosine từng vector với các vector còn lại -> Tạo thành một ma trận tương đồng. Lấy ra 1 câu truy vấn ngẫu nhiên đó với top K = 3 xem câu nào 
có kết quả tương đồng nhất.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity


def load_embedding():
    data_frame = pd.read_parquet("embedding_sentence.parquet")
    embedding = np.load("genai-section5-lab.npy")
    return data_frame, embedding


def tinh_cosine_similarity(embedding):
    return cosine_similarity(embedding)


def tinh_dot_product(embedding):
    return embedding @ embedding.T


def chuan_hoa(embedding):
    """Chuẩn hóa L2 cho mỗi vector về độ dài = 1 (Tránh lỗi chia cho 0)"""
    do_dai = np.linalg.norm(embedding, axis=1, keepdims=True)
    do_dai = np.maximum(do_dai, 1e-12)  # Tránh Division by Zero
    embedding_norm = embedding / do_dai
    return embedding_norm


def lay_top_k_tuong_dong(chi_so_cau, ma_tran_tuong_dong, k=3):
    """ 
    Trong ma trận tương đồng (là ma trận sinh ra sau khi tính cosine hay tích vô hướng cho từng vector với từng vector còn lại 
    trong ma trận embedding) tại vị trị [x][x] giá trị của nó luôn bằng 1, vì câu giống nhau nên độ tương đồng về mặt ý nghĩa sẽ
    giống nhau, chính vì vậy ta bỏ qua vị trí này.
    Cách lấy: 
    Từ ma trận tương đồng, lấy ra hàng tại vị trí đang xét, sau khi có hàng tại vị trị đang xét thì từ hàng đó lấy tiếp cột thứ
    vị trí đang xét.
    """
    #Lấy hàng thứ 'chi_so_cau' gán cho biến diem.
    diem = ma_tran_tuong_dong[chi_so_cau].copy()
    #hàng thứ 'chi_so_cau' tại vị trí 'chi_so_cau' luôn bằng 1, nên bỏ đi. Gán -9999 cho nó luôn luôn nằm cuối cùng sau khi sort.
    diem[chi_so_cau] = -999999
    
    #Đảo ngược thứ tự mảng, lấy ra từ vị trí 0 -> k
    chi_so_top = np.argsort(diem)[::-1][:k]
    
    ket_qua = []
    for i in chi_so_top:
        ket_qua.append((i, diem[i]))
    
    # ket_qua sẽ có dạng là [(index,value),(index,value)] 
    return ket_qua


def ket_qua(dataframe, indexs, cosine_matran, dot_product_matran, dot_product_matran_L2):

    for index in indexs:
        #dataframe tại dòng 'chi_so' và lấy cột 'cau'
        cau_truy_van = dataframe.iloc[index]['cau']

        #dataframe tại dòng 'chi_so' và lấy ra cột  'chu_de'
        chu_de = dataframe.iloc[index]['chu_de']
        
        print("="*80)
        print(f"Câu truy vấn [{index}]: {cau_truy_van}")
        print(f"Chủ đề gốc: {chu_de}")
        print('='*80 + '\n')
        
        # Danh sách các phép đo, LIst các Tuple. 
        phuong_phap = [
            ("Cosine Similarity (Chuẩn)", cosine_matran),
            ("Dot Product (Chưa chuẩn hóa)", dot_product_matran),
            ("Dot Product (Đã chuẩn hóa)", dot_product_matran_L2)
        ]
        
        for ten, ma_tran in phuong_phap:
            print(f"-> {ten}:")
            
            # Lấy top 3
            top_k = lay_top_k_tuong_dong(index, ma_tran, k=3)
            
            for rank, (vi_tri, gia_tri) in enumerate(top_k, 1):
                sentence = dataframe.iloc[vi_tri]['cau']
                topic = dataframe.iloc[vi_tri]['chu_de']
                print(f"{rank}. [{topic}] {sentence}... (Điểm: {gia_tri:.4f})")
            
            print("\n")

        print(f"Kết thúc câu truy vấn [{index}]: {cau_truy_van}")

            

if __name__ == "__main__":
    print("=== BẮT ĐẦU CHƯƠNG TRÌNH ===")
    
    dataframe, embeddings = load_embedding()
    
    print("[1/4] Tính Cosine Similarity...")
    cosine_matran = tinh_cosine_similarity(embeddings)
    
    print("[2/4] Tính Dot Product (thô)...")
    dot_product_matran = tinh_dot_product(embeddings)
    
    print("[3/4] Chuẩn hóa L2 embeddings...")
    embeddings_chuan_hoa_L2 = chuan_hoa(embeddings)
    
    print("[4/4] Tính Dot Product trên embedding đã chuẩn hóa...")
    dot_product_matran_L2 = tinh_dot_product(embeddings_chuan_hoa_L2)
    
    # Chọn câu truy vấn đại diện các chủ đề
    indexs = [15]
    
    ket_qua(dataframe, indexs, cosine_matran, dot_product_matran, dot_product_matran_L2)
    
